In [ ]:
# Comprehensive pandas_datareader diagnostic
import sys
import importlib

print("=== Python Environment Info ===")
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

print("\n=== Checking installed packages ===")
try:
    import pandas
    print(f"✓ pandas version: {pandas.__version__}")
except ImportError:
    print("❌ pandas not installed")

# Check both possible package names
packages_to_check = ['pandas_datareader', 'pandas-datareader']
for pkg in packages_to_check:
    try:
        spec = importlib.util.find_spec(pkg)
        if spec is not None:
            print(f"✓ {pkg} found at: {spec.origin}")
        else:
            print(f"❌ {pkg} not found")
    except ImportError:
        print(f"❌ {pkg} not found")

print("\n=== Testing imports ===")
# Try different import methods
import_attempts = [
    "import pandas_datareader",
    "import pandas_datareader.data as web", 
    "from pandas_datareader import data",
    "from pandas_datareader.data import DataReader"
]

for attempt in import_attempts:
    try:
        exec(attempt)
        print(f"✓ {attempt} - SUCCESS")
    except ImportError as e:
        print(f"❌ {attempt} - FAILED: {e}")
    except Exception as e:
        print(f"❌ {attempt} - ERROR: {e}")

print("\n=== Installation suggestion ===")
print("If imports are failing, try:")
print("1. Restart Jupyter kernel")
print("2. Run: !pip install --upgrade pandas_datareader")
print("3. Or: !pip install --upgrade pandas-datareader")

❌ Other error: deprecate_kwarg() missing 1 required positional argument: 'new_arg_name'
Check your pandas_datareader installation


In [ ]:
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import matplotlib.pyplot as plt
import datetime

Setup

In [ ]:
start_date = datetime.datetime(2000, 1, 1)
end_date = datetime.datetime(2024, 1, 1)

print("Fetching Economic Data from FRED...")
# UNRATE = Unemployment Rate (The Recession Indicator)
# RSHPCS = Advance Retail Sales: Health and Personal Care Stores (The Sales Proxy)
# PCE    = Personal Consumption Expenditures (General Spending Baseline)
fred_data = web.DataReader(['UNRATE', 'RSHPCS', 'PCE'], 'fred', start_date, end_date)

print("Fetching Stock Data from Yahoo Finance...")
# EL = Estée Lauder (The classic 'Lipstick Index' company)
# ^GSPC = S&P 500 (To compare against the general market)
tickers = ['EL', '^GSPC']
stock_data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']

Data Processing

In [ ]:
# Resample stock data to Monthly to match Economic data (taking the mean of the month)
stock_monthly = stock_data.resample('MS').mean()

# Combine everything into one DataFrame
df = pd.concat([fred_data, stock_monthly], axis=1)

# Drop NaN values (rows where data might be missing)
df = df.dropna()

# Normalize data to start at 100 (Index 100) so we can compare growth rates easily
# This helps us see relative performance over time
df_normalized = (df / df.iloc[0]) * 100

Correlation analysis

In [ ]:
correlation = df.corr()

print("\n--- CORRELATION MATRIX ---")
print(correlation[['UNRATE']])

Visualization

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Unemployment on the Left Axis (Inverted to show "Economic Health")
# Note: Usually high unemployment is "bad", so we sometimes invert it to match stock dips.
# Here we keep it normal: High Line = Bad Economy.
color = 'tab:red'
ax1.set_xlabel('Year')
ax1.set_ylabel('Unemployment Rate (%)', color=color)
ax1.plot(df.index, df['UNRATE'], color=color, linewidth=2, linestyle='--', label='Unemployment Rate (Recession)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Create a second Y-axis for the "Lipstick" proxies
ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('Stock Price & Retail Sales (Normalized)', color=color2)

# Plot Estee Lauder Stock
ax2.plot(df.index, df_normalized['EL'], color='purple', linewidth=2, label='Estée Lauder Stock (Lipstick Proxy)')

# Plot Health & Personal Care Sales (Actual Sales Data)
ax2.plot(df.index, df_normalized['RSHPCS'], color='green', linewidth=2, alpha=0.6, label='Health & Beauty Sales')

# Add Legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.title('The Lipstick Effect: Recessions vs. Beauty Sales (2000-2024)')
plt.show()